In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abhinandm5856/customer-behaviour/customer_shopping_behavior.csv


In [2]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/abhinandm5856/customer-behaviour/customer_shopping_behavior.csv')

In [18]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,customer_segment,purchase_frequency_days,purchase_frequency_group
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Yes,Express,Yes,14,Venmo,Fortnightly,Senior,Medium Value,14,Frequent
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Yes,Express,Yes,2,Cash,Fortnightly,Youth,Medium Value,14,Frequent
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Adult,High Value,7,Frequent
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Yes,Next Day Air,Yes,49,PayPal,Weekly,Adult,High Value,7,Frequent
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Yes,Free Shipping,Yes,31,PayPal,Annually,Adult,Medium Value,365,Rare


In [4]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [5]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [6]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(
    lambda x: x.fillna(x.median()))

In [7]:
df.columns


Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Promo Code Used', 'Previous Purchases',
       'Payment Method', 'Frequency of Purchases'],
      dtype='object')

In [8]:
df.columns = df.columns.str.lower()

df.columns = df.columns.str.replace(' ','_')

df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [9]:
bins = [0, 19, 50, 100]

labels = ['Youth', 'Adult', 'Senior']

df['age_group'] = pd.cut(
    df['age'],
    bins=bins,
    labels=labels
)

In [10]:
df['age_group'].value_counts()

age_group
Adult     2274
Senior    1476
Youth      150
Name: count, dtype: int64

In [11]:
df['customer_segment'] = pd.cut(
    df['purchase_amount'],
    bins=[0, 40, 70, 100],
    labels=['Low Value', 'Medium Value', 'High Value']
)

In [12]:
df[['age', 'age_group', 'purchase_amount', 'customer_segment']].head()

,age,age_group,purchase_amount,customer_segment
0,55,Senior,53,Medium Value
1,19,Youth,64,Medium Value
2,50,Adult,73,High Value
3,21,Adult,90,High Value
4,45,Adult,49,Medium Value


In [13]:
# Create purchase_frequency_days column

frequency_mapping = {
    'Weekly': 7,
    'Fortnightly': 14,
    'Bi-Weekly': 14,
    'Monthly': 30,
    'Quarterly': 90,
    'Every 3 Months': 90,
    'Annually': 365
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [14]:
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


In [15]:
# Create purchase_frequency_group column

df['purchase_frequency_group'] = pd.cut(
    df['purchase_frequency_days'],
    bins=[0, 15, 60, 400],
    labels=['Frequent', 'Regular', 'Rare']
)

In [16]:
df[['frequency_of_purchases',
    'purchase_frequency_days',
    'purchase_frequency_group']].head(10)

,frequency_of_purchases,purchase_frequency_days,purchase_frequency_group
0,Fortnightly,14,Frequent
1,Fortnightly,14,Frequent
2,Weekly,7,Frequent
3,Weekly,7,Frequent
4,Annually,365,Rare
5,Weekly,7,Frequent
6,Quarterly,90,Rare
7,Weekly,7,Frequent
8,Annually,365,Rare
9,Quarterly,90,Rare


In [17]:
df = df.drop('promo_code_used', axis=1)

In [34]:
df.to_csv('customer_shopping_cleaned.csv', index=False)